In [1]:
import os
import time
import requests
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv
from tqdm import tqdm

In [12]:
load_dotenv()
OPENAQ_API_KEY = os.getenv("OPENAQ_API_KEY")

In [51]:
# Date range for the project
START_DATE = "2016-01-01"
END_DATE   = "2025-12-31"

DATA_DIR_RAW = "../data/raw"
DATA_DIR_PROCESSED = "../data/processed"

os.makedirs(DATA_DIR_RAW,       exist_ok=True)
os.makedirs(DATA_DIR_PROCESSED, exist_ok=True)

In [22]:
url = "https://api.openaq.org/v3/locations"
params = {
    "country_id": "IN",
    "city":       "Delhi",
    "limit":      100,
    "page":       1
}
headers = {"X-API-Key": OPENAQ_API_KEY}

response = requests.get(url, params=params, headers=headers, timeout=30)
data = response.json()

In [23]:
# Delhi bounding box
DELHI_BOUNDS = {
    "lat_min": 28.40,
    "lat_max": 28.88,
    "lon_min": 76.84,
    "lon_max": 77.35
}

headers = {"X-API-Key": OPENAQ_API_KEY}
stations = []
page = 1

while True:
    url = "https://api.openaq.org/v3/locations"
    params = {
        "bbox":     f"{DELHI_BOUNDS['lon_min']},{DELHI_BOUNDS['lat_min']},{DELHI_BOUNDS['lon_max']},{DELHI_BOUNDS['lat_max']}",
        "limit":    100,
        "page":     page
    }

    response = requests.get(url, params=params, headers=headers, timeout=30)
    data = response.json()
    results = data.get("results", [])
    
    if not results:
        break

    for loc in results:
        for sensor in loc.get("sensors", []):
            param_name = sensor["parameter"]["name"]
            if param_name in ["pm25", "pm10"]:
                stations.append({
                    "location_id":   loc["id"],
                    "location_name": loc["name"],
                    "sensor_id":     sensor["id"],
                    "parameter":     param_name,
                    "lat":           loc.get("coordinates", {}).get("latitude"),
                    "lon":           loc.get("coordinates", {}).get("longitude"),
                })
    
    # Stop if we've got all pages
    found = data["meta"].get("found", "0")
    total = int(found) if found != ">1000" else 1000
    if page * 100 >= total:
        break
    page += 1

stations_df = pd.DataFrame(stations).drop_duplicates(subset=["location_id", "parameter"])
print(f"Found {len(stations_df)} station-parameter combos in Delhi")
print(stations_df[["location_name", "parameter", "lat", "lon"]].to_string())

Found 165 station-parameter combos in Delhi
                                        location_name parameter        lat        lon
0        Delhi Technological University, Delhi - CPCB      pm25  28.744000  77.120000
1                                         IGI Airport      pm10  28.560000  77.094000
2                                         IGI Airport      pm25  28.560000  77.094000
3                                         Civil Lines      pm10  28.678700  77.226200
4                                         Civil Lines      pm25  28.678700  77.226200
5                             R K Puram, Delhi - DPCC      pm10  28.563262  77.186937
7                             R K Puram, Delhi - DPCC      pm25  28.563262  77.186937
9                          Punjabi Bagh, Delhi - DPCC      pm10  28.674045  77.131023
11                         Punjabi Bagh, Delhi - DPCC      pm25  28.674045  77.131023
13                    Income Tax Office, Delhi - CPCB      pm10  28.623500  77.249400
14        

In [49]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def fetch_sensor_daily_paginated(row):
    sensor_id     = row["sensor_id"]
    location_id   = row["location_id"]
    location_name = row["location_name"]
    parameter     = row["parameter"]

    url     = f"https://api.openaq.org/v3/sensors/{sensor_id}/days"
    headers = {"X-API-Key": OPENAQ_API_KEY}
    all_rows = []
    page = 1

    while True:
        params = {
            "date_from": START_DATE,
            "date_to":   END_DATE,
            "limit":     1000,
            "page":      page,
        }
        try:
            resp    = requests.get(url, params=params, headers=headers, timeout=30)
            body    = resp.json()
            results = body.get("results", [])
            if not results:
                break
            for r in results:
                all_rows.append({
                    "date":          r["period"]["datetimeFrom"]["utc"][:10],
                    "value":         r["value"],
                    "sensor_id":     sensor_id,
                    "location_id":   location_id,
                    "location_name": location_name,
                    "parameter":     parameter,
                })
            # Stop if this page had fewer than 1000 — means last page
            if len(results) < 1000:
                break
            page += 1
        except Exception as e:
            print(f"⚠️ sensor {sensor_id} page {page}: {e}")
            break

    return all_rows

# Parallel fetch
all_measurements = []
rows_list = [row for _, row in stations_df.iterrows()]

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(fetch_sensor_daily_paginated, row): row for row in rows_list}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Fetching"):
        try:
            all_measurements.extend(future.result())
        except Exception as e:
            print(f"⚠️ Error: {e}")

aq_raw = pd.DataFrame(all_measurements)
aq_raw["date"] = pd.to_datetime(aq_raw["date"])

print(f"✅ Total rows: {len(aq_raw)}")
print(f"📅 Range: {aq_raw['date'].min()} → {aq_raw['date'].max()}")
print(f"📍 Unique sensors: {aq_raw['sensor_id'].nunique()}")
print(f"🔁 Duplicates: {aq_raw.duplicated(subset=['date','sensor_id']).sum()}")

# Save raw
aq_raw.to_csv(f"{DATA_DIR}/openaq_raw.csv", index=False)
aq_raw.head()

Fetching:   0%|          | 0/165 [00:00<?, ?it/s]

Fetching: 100%|██████████| 165/165 [00:06<00:00, 26.55it/s]


✅ Total rows: 23543
📅 Range: 2016-01-29 00:00:00 → 2025-12-30 00:00:00
📍 Unique sensors: 48
🔁 Duplicates: 0


,date,value,sensor_id,location_id,location_name,parameter
0,2025-02-18,178.0,12234786,17,"R K Puram, Delhi - DPCC",pm10
1,2025-02-19,105.0,12234786,17,"R K Puram, Delhi - DPCC",pm10
2,2025-02-20,164.0,12234786,17,"R K Puram, Delhi - DPCC",pm10
3,2025-02-21,138.0,12234786,17,"R K Puram, Delhi - DPCC",pm10
4,2025-02-22,142.0,12234786,17,"R K Puram, Delhi - DPCC",pm10


In [34]:
# City-wide daily average across all stations per parameter
aq_daily = (
    aq_raw
    .groupby(["date", "parameter"])["value"]
    .mean()
    .reset_index()
    .pivot(index="date", columns="parameter", values="value")
    .reset_index()
)
aq_daily.columns.name = None
aq_daily.rename(columns={"pm25": "pm25_avg", "pm10": "pm10_avg"}, inplace=True)
aq_daily["date"] = pd.to_datetime(aq_daily["date"])

# Save both
aq_raw.to_csv(f"{DATA_DIR}/openaq_raw.csv", index=False)
aq_daily.to_csv(f"{DATA_DIR}/air_quality_daily.csv", index=False)

print(f"✅ City-level daily shape: {aq_daily.shape}")
print(f"📅 Range: {aq_daily['date'].min()} → {aq_daily['date'].max()}")
print(f"Missing pm25: {aq_daily['pm25_avg'].isna().sum()} days")
print(f"Missing pm10: {aq_daily['pm10_avg'].isna().sum()} days")
aq_daily.head(10)

✅ City-level daily shape: (2233, 3)
📅 Range: 2016-01-29 00:00:00 → 2025-12-30 00:00:00
Missing pm25: 0 days
Missing pm10: 41 days


,date,pm10_avg,pm25_avg
0,2016-01-29,NaN,356.00
1,2016-01-30,NaN,203.00
2,2016-01-31,NaN,124.00
3,2016-02-01,NaN,136.00
4,2016-02-02,NaN,148.00
5,2016-02-03,NaN,137.00
6,2016-02-04,489.333333,207.20
7,2016-02-05,406.666667,201.40
8,2016-02-06,256.000000,160.25
9,2016-02-07,274.333333,131.64


In [35]:
import openmeteo_requests
import requests_cache
from retry_requests import retry

# Setup cached session
cache_session = requests_cache.CachedSession(".cache", expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
om = openmeteo_requests.Client(session=retry_session)

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude":   28.6139,
    "longitude":  77.2090,
    "start_date": START_DATE,
    "end_date":   END_DATE,
    "daily": [
        "temperature_2m_max",
        "temperature_2m_min",
        "temperature_2m_mean",
        "wind_speed_10m_max",
        "wind_speed_10m_mean",
        "relative_humidity_2m_mean",
        "precipitation_sum",
        "visibility_mean",
    ],
    "timezone": "Asia/Kolkata"
}

responses = om.weather_api(url, params=params)
response  = responses[0]
daily     = response.Daily()

weather_df = pd.DataFrame({
    "date":           pd.date_range(
                          start=pd.to_datetime(daily.Time(),    unit="s", utc=True).tz_localize(None),
                          end=pd.to_datetime(daily.TimeEnd(),   unit="s", utc=True).tz_localize(None),
                          freq=pd.Timedelta(seconds=daily.Interval()),
                          inclusive="left"
                      ),
    "temp_max":       daily.Variables(0).ValuesAsNumpy(),
    "temp_min":       daily.Variables(1).ValuesAsNumpy(),
    "temp_mean":      daily.Variables(2).ValuesAsNumpy(),
    "wind_speed_max": daily.Variables(3).ValuesAsNumpy(),
    "wind_speed_mean":daily.Variables(4).ValuesAsNumpy(),
    "humidity_mean":  daily.Variables(5).ValuesAsNumpy(),
    "precipitation":  daily.Variables(6).ValuesAsNumpy(),
    "visibility":     daily.Variables(7).ValuesAsNumpy(),
})

weather_df["date"] = pd.to_datetime(weather_df["date"])

weather_df.to_csv(f"{DATA_DIR}/weather_daily.csv", index=False)

print(f"✅ Weather data shape: {weather_df.shape}")
print(f"📅 Range: {weather_df['date'].min()} → {weather_df['date'].max()}")
print(f"Missing values:\n{weather_df.isna().sum()}")
weather_df.head()

✅ Weather data shape: (3653, 9)
📅 Range: 2015-12-31 18:30:00 → 2025-12-30 18:30:00
Missing values:
date                  0
temp_max              0
temp_min              0
temp_mean             0
wind_speed_max        0
wind_speed_mean       0
humidity_mean         0
precipitation         0
visibility         3653
dtype: int64


,date,temp_max,temp_min,temp_mean,wind_speed_max,wind_speed_mean,humidity_mean,precipitation,visibility
0,2015-12-31 18:30:00,22.297501,9.297500,14.937081,9.693295,6.144559,68.453026,0.0,NaN
1,2016-01-01 18:30:00,22.047501,7.297500,14.412084,10.739832,7.835854,65.795128,0.0,NaN
2,2016-01-02 18:30:00,22.647499,8.747499,14.968333,13.854155,10.166673,62.526402,0.0,NaN
3,2016-01-03 18:30:00,22.897499,8.397500,16.064165,9.957108,6.039811,65.527191,0.0,NaN
4,2016-01-04 18:30:00,24.497499,12.497499,18.055834,8.825508,5.550406,68.818520,0.0,NaN


In [36]:
# Fix 1: Drop visibility (unavailable), fix timestamp to date only
weather_df = weather_df.drop(columns=["visibility"])
weather_df["date"] = weather_df["date"].dt.normalize()  # strips time component

# Fix 2: Trim to only dates where we have AQ data
weather_df = weather_df[
    (weather_df["date"] >= aq_daily["date"].min()) &
    (weather_df["date"] <= aq_daily["date"].max())
]

# Overwrite saved file
weather_df.to_csv(f"{DATA_DIR}/weather_daily.csv", index=False)

print(f"✅ Weather shape after trim: {weather_df.shape}")
print(f"📅 Range: {weather_df['date'].min()} → {weather_df['date'].max()}")
print(f"Missing values:\n{weather_df.isna().sum()}")
weather_df.head()

✅ Weather shape after trim: (3624, 8)
📅 Range: 2016-01-29 00:00:00 → 2025-12-30 00:00:00
Missing values:
date               0
temp_max           0
temp_min           0
temp_mean          0
wind_speed_max     0
wind_speed_mean    0
humidity_mean      0
precipitation      0
dtype: int64


,date,temp_max,temp_min,temp_mean,wind_speed_max,wind_speed_mean,humidity_mean,precipitation
29,2016-01-29,25.497499,13.747499,19.370419,8.396570,5.723620,72.746208,0.5
30,2016-01-30,22.147499,12.847500,17.372501,14.007655,10.711514,77.328606,0.0
31,2016-01-31,22.097500,9.697500,15.164165,12.496718,10.017783,64.969994,0.0
32,2016-02-01,20.947500,7.447500,14.197501,15.349684,9.648987,58.050755,0.0
33,2016-02-02,21.547501,8.347500,14.468334,20.620804,12.373805,53.275467,0.0


In [43]:
from pytrends.request import TrendReq
from dateutil.relativedelta import relativedelta
import time

trends_client = TrendReq(hl="en-IN", tz=330, timeout=(10, 25)) 

def fetch_trends_chunked(keyword, start, end):
    all_frames = []
    chunk_start = datetime.strptime(start, "%Y-%m-%d")
    chunk_end   = datetime.strptime(end,   "%Y-%m-%d")

    while chunk_start < chunk_end:
        year_end  = min(chunk_start + relativedelta(years=1), chunk_end)
        timeframe = f"{chunk_start.strftime('%Y-%m-%d')} {year_end.strftime('%Y-%m-%d')}"

        try:
            trends_client.build_payload([keyword], geo="IN-DL", timeframe=timeframe)  # ✅
            df = trends_client.interest_over_time()                                    # ✅
            if not df.empty:
                all_frames.append(df[[keyword]])
            time.sleep(3)
        except Exception as e:
            print(f"⚠️ {keyword} chunk {chunk_start.year}: {e}")

        chunk_start = year_end

    return pd.concat(all_frames) if all_frames else pd.DataFrame()

keywords = [
    "air pollution Delhi",
    "N95 mask",
    "air purifier",
    "breathing problem",
    "AQI Delhi"
]

trends_frames = []
for kw in keywords:
    print(f"Fetching: {kw}")
    df = fetch_trends_chunked(kw, START_DATE, END_DATE)
    if not df.empty:
        df = df.rename(columns={kw: kw.replace(" ", "_")})
        trends_frames.append(df)
        print(f"  ✅ {len(df)} weeks")
    else:
        print(f"  ❌ No data")

trends_df = pd.concat(trends_frames, axis=1).reset_index()
trends_df.rename(columns={"date": "week"}, inplace=True)
trends_df["week"] = pd.to_datetime(trends_df["week"])
trends_df = trends_df.drop_duplicates(subset=["week"]).sort_values("week").reset_index(drop=True)

trends_df.to_csv(f"{DATA_DIR}/google_trends_weekly.csv", index=False)

print(f"\n✅ Google Trends shape: {trends_df.shape}")
print(f"📅 Range: {trends_df['week'].min()} → {trends_df['week'].max()}")
trends_df.head()

Fetching: air pollution Delhi
  ✅ 532 weeks
Fetching: N95 mask
  ✅ 532 weeks
Fetching: air purifier
  ✅ 532 weeks
Fetching: breathing problem
  ✅ 532 weeks
Fetching: AQI Delhi
  ✅ 532 weeks

✅ Google Trends shape: (523, 6)
📅 Range: 2015-12-27 00:00:00 → 2025-12-28 00:00:00


,week,air_pollution_Delhi,N95_mask,air_purifier,breathing_problem,AQI_Delhi
0,2015-12-27,10,0,6,37,13
1,2016-01-03,18,0,8,61,11
2,2016-01-10,15,0,6,0,0
3,2016-01-17,11,0,6,0,0
4,2016-01-24,5,0,4,0,0


In [44]:
# Cell A — Shape and columns of all datasets
for fname in ["openaq_raw.csv", "air_quality_daily.csv", "weather_daily.csv", "google_trends_weekly.csv"]:
    df = pd.read_csv(f"{DATA_DIR}/{fname}")
    print(f"\n{'='*50}")
    print(f"📄 {fname}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"Dtypes:\n{df.dtypes}")


📄 openaq_raw.csv
Shape: (23430, 6)
Columns: ['date', 'value', 'sensor_id', 'location_id', 'location_name', 'parameter']
Dtypes:
date              object
value            float64
sensor_id          int64
location_id        int64
location_name     object
parameter         object
dtype: object

📄 air_quality_daily.csv
Shape: (2233, 3)
Columns: ['date', 'pm10_avg', 'pm25_avg']
Dtypes:
date         object
pm10_avg    float64
pm25_avg    float64
dtype: object

📄 weather_daily.csv
Shape: (3624, 8)
Columns: ['date', 'temp_max', 'temp_min', 'temp_mean', 'wind_speed_max', 'wind_speed_mean', 'humidity_mean', 'precipitation']
Dtypes:
date                object
temp_max           float64
temp_min           float64
temp_mean          float64
wind_speed_max     float64
wind_speed_mean    float64
humidity_mean      float64
precipitation      float64
dtype: object

📄 google_trends_weekly.csv
Shape: (523, 6)
Columns: ['week', 'air_pollution_Delhi', 'N95_mask', 'air_purifier', 'breathing_problem', 'AQI_

In [45]:
# Cell B — Missing values and basic stats for each
for fname in ["openaq_raw.csv", "air_quality_daily.csv", "weather_daily.csv", "google_trends_weekly.csv"]:
    df = pd.read_csv(f"{DATA_DIR}/{fname}")
    print(f"\n{'='*50}")
    print(f"📄 {fname}")
    print(f"Missing values:\n{df.isna().sum()}")
    print(f"\nSample rows:")
    print(df.head(3).to_string())


📄 openaq_raw.csv
Missing values:
date             0
value            4
sensor_id        0
location_id      0
location_name    0
parameter        0
dtype: int64

Sample rows:
         date  value  sensor_id  location_id            location_name parameter
0  2025-02-18  178.0   12234786           17  R K Puram, Delhi - DPCC      pm10
1  2025-02-19  105.0   12234786           17  R K Puram, Delhi - DPCC      pm10
2  2025-02-20  164.0   12234786           17  R K Puram, Delhi - DPCC      pm10

📄 air_quality_daily.csv
Missing values:
date         0
pm10_avg    41
pm25_avg     0
dtype: int64

Sample rows:
         date  pm10_avg  pm25_avg
0  2016-01-29       NaN     356.0
1  2016-01-30       NaN     203.0
2  2016-01-31       NaN     124.0

📄 weather_daily.csv
Missing values:
date               0
temp_max           0
temp_min           0
temp_mean          0
wind_speed_max     0
wind_speed_mean    0
humidity_mean      0
precipitation      0
dtype: int64

Sample rows:
         date  temp_max 

In [52]:
import shutil
raw_files = ["openaq_raw.csv", "air_quality_daily.csv", "weather_daily.csv", "google_trends_weekly.csv"]

for fname in raw_files:
    old_path = f"../data/{fname}"
    new_path = f"{DATA_DIR_RAW}/{fname}"
    if os.path.exists(old_path):
        shutil.move(old_path, new_path)
        print(f"✅ Moved {fname} → data/raw/")
    elif os.path.exists(new_path):
        print(f"ℹ️  Already in raw/: {fname}")
    else:
        print(f"❌ Not found: {fname}")

✅ Moved openaq_raw.csv → data/raw/
✅ Moved air_quality_daily.csv → data/raw/
✅ Moved weather_daily.csv → data/raw/
✅ Moved google_trends_weekly.csv → data/raw/


In [53]:
import pandas as pd
import numpy as np

# Load all
aq_raw     = pd.read_csv(f"{DATA_DIR_RAW}/openaq_raw.csv")
aq_daily   = pd.read_csv(f"{DATA_DIR_RAW}/air_quality_daily.csv")
weather_df = pd.read_csv(f"{DATA_DIR_RAW}/weather_daily.csv")
trends_df  = pd.read_csv(f"{DATA_DIR_RAW}/google_trends_weekly.csv")

# Parse dates
aq_raw["date"]     = pd.to_datetime(aq_raw["date"])
aq_daily["date"]   = pd.to_datetime(aq_daily["date"])
weather_df["date"] = pd.to_datetime(weather_df["date"])
trends_df["week"]  = pd.to_datetime(trends_df["week"])


print(f"aq_raw date dtype:     {aq_raw['date'].dtype}")
print(f"aq_daily date dtype:   {aq_daily['date'].dtype}")
print(f"weather date dtype:    {weather_df['date'].dtype}")
print(f"trends week dtype:     {trends_df['week'].dtype}")

aq_raw date dtype:     datetime64[ns]
aq_daily date dtype:   datetime64[ns]
weather date dtype:    datetime64[ns]
trends week dtype:     datetime64[ns]


In [54]:
print(f"Rows before: {len(aq_raw)}")
print(f"Missing values:\n{aq_raw[aq_raw['value'].isna()][['date','location_name','parameter']]}")

# Impute 4 missing values with mean of same sensor + parameter
aq_raw["value"] = aq_raw.groupby(["sensor_id", "parameter"])["value"].transform(
    lambda x: x.fillna(x.mean())
)

print(f"\nMissing after imputation: {aq_raw['value'].isna().sum()}")

# Drop physically impossible values
print(f"\nOutlier check — PM2.5 > 1000: {(aq_raw[aq_raw['parameter']=='pm25']['value'] > 1000).sum()}")
print(f"Outlier check — PM10  > 1500: {(aq_raw[aq_raw['parameter']=='pm10']['value'] > 1500).sum()}")
print(f"Negative values: {(aq_raw['value'] < 0).sum()}")

aq_raw = aq_raw[(aq_raw["value"] >= 0)]
aq_raw.loc[aq_raw["parameter"] == "pm25", "value"] = aq_raw.loc[aq_raw["parameter"] == "pm25", "value"].clip(upper=1000)
aq_raw.loc[aq_raw["parameter"] == "pm10", "value"] = aq_raw.loc[aq_raw["parameter"] == "pm10", "value"].clip(upper=1500)

print(f"Rows after: {len(aq_raw)}")
aq_raw.to_csv(f"{DATA_DIR_PROCESSED}/openaq_raw.csv", index=False)
print("✅ openaq_raw cleaned")

Rows before: 23543
Missing values:
           date                  location_name parameter
5903 2016-03-18  US Diplomatic Post: New Delhi      pm25
5904 2016-03-19  US Diplomatic Post: New Delhi      pm25
6009 2016-07-02  US Diplomatic Post: New Delhi      pm25
6010 2016-07-03  US Diplomatic Post: New Delhi      pm25

Missing after imputation: 0

Outlier check — PM2.5 > 1000: 6
Outlier check — PM10  > 1500: 4
Negative values: 38
Rows after: 23505
✅ openaq_raw cleaned


In [60]:
print(f"Missing pm10 before: {aq_daily['pm10_avg'].isna().sum()}")
print(f"Missing pm25 before: {aq_daily['pm25_avg'].isna().sum()}")

# Check if missing pm10 days have pm25 — i.e. are they early 2016 days?
missing_pm10 = aq_daily[aq_daily["pm10_avg"].isna()]
print(f"\nMissing pm10 date range: {missing_pm10['date'].min()} → {missing_pm10['date'].max()}")

# Interpolate — linear is fine for 41 scattered gaps in a continuous signal
aq_daily["pm10_avg"] = aq_daily["pm10_avg"].interpolate(method="linear", limit=7)

# For any remaining gaps > 7 consecutive days, use seasonal median (same month)
aq_daily["month"] = aq_daily["date"].dt.month
monthly_pm10 = aq_daily.groupby("month")["pm10_avg"].median()
aq_daily["pm10_avg"] = aq_daily.apply(
    lambda row: monthly_pm10[row["month"]] if pd.isna(row["pm10_avg"]) else row["pm10_avg"],
    axis=1
)
aq_daily = aq_daily.drop(columns=["month"])

print(f"\nMissing pm10 after:  {aq_daily['pm10_avg'].isna().sum()}")
print(f"Missing pm25 after:  {aq_daily['pm25_avg'].isna().sum()}")

Missing pm10 before: 41
Missing pm25 before: 0

Missing pm10 date range: 2016-01-29 00:00:00 → 2021-05-03 00:00:00

Missing pm10 after:  0
Missing pm25 after:  0


In [62]:
# India CPCB AQI breakpoints for PM2.5
def compute_aqi_pm25(pm25):
    if pd.isna(pm25):
        return np.nan
    breakpoints = [
        (0,   30,   0,   50),
        (30,  60,   51,  100),
        (60,  90,   101, 200),
        (90,  120,  201, 300),
        (120, 250,  301, 400),
        (250, 500,  401, 500),
    ]
    for bp_lo, bp_hi, aqi_lo, aqi_hi in breakpoints:
        if bp_lo <= pm25 <= bp_hi:
            return round(((aqi_hi - aqi_lo) / (bp_hi - bp_lo)) * (pm25 - bp_lo) + aqi_lo)
    return 500  # cap

aq_daily["aqi"] = aq_daily["pm25_avg"].apply(compute_aqi_pm25)

# AQI category labels
def aqi_category(aqi):
    if pd.isna(aqi):   return np.nan
    if aqi <= 50:      return "Good"
    if aqi <= 100:     return "Satisfactory"
    if aqi <= 200:     return "Moderate"
    if aqi <= 300:     return "Poor"
    if aqi <= 400:     return "Very Poor"
    return "Severe"

aq_daily["aqi_category"] = aq_daily["aqi"].apply(aqi_category)

print("AQI distribution:")
print(aq_daily["aqi_category"].value_counts())
print(f"\nAQI range: {aq_daily['aqi'].min():.0f} → {aq_daily['aqi'].max():.0f}")
aq_daily.to_csv(f"{DATA_DIR_PROCESSED}/air_quality_daily.csv", index=False)
print("✅ AQI computed and saved")

AQI distribution:
aqi_category
Very Poor       649
Satisfactory    573
Moderate        389
Poor            269
Good            205
Severe          148
Name: count, dtype: int64

AQI range: 13 → 500
✅ AQI computed and saved


In [63]:
print(f"Weather rows before trim: {len(weather_df)}")

aq_start = aq_daily["date"].min()
aq_end   = aq_daily["date"].max()

weather_df = weather_df[
    (weather_df["date"] >= aq_start) &
    (weather_df["date"] <= aq_end)
].reset_index(drop=True)

print(f"Weather rows after trim:  {len(weather_df)}")
print(f"📅 Range: {weather_df['date'].min()} → {weather_df['date'].max()}")

# Round to 2 decimal places — Open-Meteo has excessive precision
for col in ["temp_max","temp_min","temp_mean","wind_speed_max","wind_speed_mean","humidity_mean","precipitation"]:
    weather_df[col] = weather_df[col].round(2)

weather_df.to_csv(f"{DATA_DIR_PROCESSED}/weather_daily.csv", index=False)
print("✅ Weather trimmed and saved")

Weather rows before trim: 3624
Weather rows after trim:  3624
📅 Range: 2016-01-29 00:00:00 → 2025-12-30 00:00:00
✅ Weather trimmed and saved


In [65]:
# Forward-fill each week's value to its 7 days
daily_rows = []

for _, row in trends_df.iterrows():
    for i in range(7):
        daily_rows.append({
            "date":                row["week"] + pd.Timedelta(days=i),
            "air_pollution_Delhi": row["air_pollution_Delhi"],
            "N95_mask":            row["N95_mask"],
            "air_purifier":        row["air_purifier"],
            "breathing_problem":   row["breathing_problem"],
            "AQI_Delhi":           row["AQI_Delhi"],
        })

trends_daily = pd.DataFrame(daily_rows)

# Trim to AQ date range
trends_daily = trends_daily[
    (trends_daily["date"] >= aq_start) &
    (trends_daily["date"] <= aq_end)
].reset_index(drop=True)

trends_daily.to_csv(f"{DATA_DIR_PROCESSED}/google_trends_daily.csv", index=False)

print(f"✅ Trends expanded: {len(trends_df)} weeks → {len(trends_daily)} days")
print(f"📅 Range: {trends_daily['date'].min()} → {trends_daily['date'].max()}")

✅ Trends expanded: 523 weeks → 3624 days
📅 Range: 2016-01-29 00:00:00 → 2025-12-30 00:00:00


In [66]:
# Left join on AQ dates — AQ is the primary dataset
master = aq_daily.copy()
master = master.merge(weather_df,    on="date", how="left")
master = master.merge(trends_daily,  on="date", how="left")

master = master.sort_values("date").reset_index(drop=True)

# Add time features — useful for all 10 research questions
master["year"]    = master["date"].dt.year
master["month"]   = master["date"].dt.month
master["season"]  = master["month"].map({
    12: "Winter", 1: "Winter",  2: "Winter",
    3:  "Spring", 4: "Spring",  5: "Spring",
    6:  "Monsoon",7: "Monsoon", 8: "Monsoon",
    9:  "Post-Monsoon", 10: "Post-Monsoon", 11: "Post-Monsoon"
})

master.to_csv(f"{DATA_DIR}/master_daily.csv", index=False)

print(f"✅ Master dataset shape: {master.shape}")
print(f"📅 Range: {master['date'].min()} → {master['date'].max()}")
print(f"Columns: {list(master.columns)}")
print(f"\nMissing values:\n{master.isna().sum()}")
master.head()

✅ Master dataset shape: (2233, 20)
📅 Range: 2016-01-29 00:00:00 → 2025-12-30 00:00:00
Columns: ['date', 'pm10_avg', 'pm25_avg', 'aqi', 'aqi_category', 'temp_max', 'temp_min', 'temp_mean', 'wind_speed_max', 'wind_speed_mean', 'humidity_mean', 'precipitation', 'air_pollution_Delhi', 'N95_mask', 'air_purifier', 'breathing_problem', 'AQI_Delhi', 'year', 'month', 'season']

Missing values:
date                   0
pm10_avg               0
pm25_avg               0
aqi                    0
aqi_category           0
temp_max               0
temp_min               0
temp_mean              0
wind_speed_max         0
wind_speed_mean        0
humidity_mean          0
precipitation          0
air_pollution_Delhi    0
N95_mask               0
air_purifier           0
breathing_problem      0
AQI_Delhi              0
year                   0
month                  0
season                 0
dtype: int64


,date,pm10_avg,pm25_avg,aqi,aqi_category,temp_max,temp_min,temp_mean,wind_speed_max,wind_speed_mean,humidity_mean,precipitation,air_pollution_Delhi,N95_mask,air_purifier,breathing_problem,AQI_Delhi,year,month,season
0,2016-01-29,272.375000,356.0,443,Severe,25.50,13.75,19.37,8.40,5.72,72.75,0.5,5,0,4,0,0,2016,1,Winter
1,2016-01-30,272.375000,203.0,364,Very Poor,22.15,12.85,17.37,14.01,10.71,77.33,0.0,5,0,4,0,0,2016,1,Winter
2,2016-01-31,272.375000,124.0,304,Very Poor,22.10,9.70,15.16,12.50,10.02,64.97,0.0,5,0,6,56,0,2016,1,Winter
3,2016-02-01,257.666667,136.0,313,Very Poor,20.95,7.45,14.20,15.35,9.65,58.05,0.0,5,0,6,56,0,2016,2,Winter
4,2016-02-02,257.666667,148.0,322,Very Poor,21.55,8.35,14.47,20.62,12.37,53.28,0.0,5,0,6,56,0,2016,2,Winter


In [73]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats

In [75]:
import os
import plotly.io as pio

# Create a folder to save all charts
FIGURES_DIR = "../figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

def show(fig, filename):
    """Save figure as HTML and display inline."""
    path = f"{FIGURES_DIR}/{filename}.html"
    fig.write_html(path)
    print(f"✅ Saved → {path}")
    fig.write_image(f"{FIGURES_DIR}/{filename}.png", width=1200, height=500, scale=2)
    display(fig)

In [77]:
heatmap_data = master.groupby(["year", "month"])["aqi"].mean().reset_index()
heatmap_pivot = heatmap_data.pivot(index="year", columns="month", values="aqi")

month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

fig = px.imshow(
    heatmap_pivot,
    labels=dict(x="Month", y="Year", color="Avg AQI"),
    x=month_labels,
    y=heatmap_pivot.index.tolist(),
    color_continuous_scale=[
        [0,   "#2ecc71"],
        [0.2, "#f1c40f"],
        [0.4, "#e67e22"],
        [0.6, "#e74c3c"],
        [1.0, "#7b241c"],
    ],
    aspect="auto",
    text_auto=".0f"
)

fig.update_layout(
    title="Monthly Average AQI Heatmap — Delhi (2016–2025)",
    height=420,
    xaxis_title="",
    yaxis_title=""
)
fig.update_xaxes(side="top")
show(fig, "01_pm25_trend")

print("""
Insight: The heatmap reveals a consistent pollution calendar — Nov–Jan is universally
dark red across all years, while Jul–Aug shows relief (green/yellow). October marks
the onset of severe AQI every single year, confirming crop-burning season and winter
inversion as primary drivers.
""")

✅ Saved → ../figures/01_pm25_trend.html


ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido


In [81]:
master_sorted = master.sort_values("date")

# Monthly average for cleaner signal
monthly = master_sorted.resample("ME", on="date").agg({
    "pm25_avg":            "mean",
    "air_pollution_Delhi": "mean",
    "AQI_Delhi":           "mean"
}).reset_index()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(
    x=monthly["date"],
    y=monthly["pm25_avg"],
    name="Monthly Avg PM2.5",
    line=dict(color="#e74c3c", width=2)
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=monthly["date"],
    y=monthly["air_pollution_Delhi"],
    name="Search: 'air pollution Delhi'",
    line=dict(color="#2980b9", width=2, dash="dot")
), secondary_y=True)

fig.add_trace(go.Scatter(
    x=monthly["date"],
    y=monthly["AQI_Delhi"],
    name="Search: 'AQI Delhi'",
    line=dict(color="#8e44ad", width=2, dash="dash")
), secondary_y=True)

fig.update_layout(
    title="Public Search Interest vs Actual PM2.5 Levels — Does Awareness Follow Pollution?",
    height=480,
    hovermode="x unified"
)
fig.update_yaxes(title_text="PM2.5 (µg/m³)",       secondary_y=False)
fig.update_yaxes(title_text="Google Trends Index",  secondary_y=True)

show(fig, "08_trends_vs_pm25")

print("""
Insight: Search interest for 'air pollution Delhi' and 'AQI Delhi' closely follows
actual PM2.5 spikes, particularly during Oct–Nov each year. However, public attention
decays faster than pollution levels — searches drop within 2–4 weeks even when PM2.5
remains elevated, suggesting short-lived public concern relative to sustained exposure.
""")

✅ Saved → ../figures/08_trends_vs_pm25.html


ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido
